# Anatomical gait information

**Purpose:** Compare five fixed anatomical waveform channel groups on matched held-out trials.

**Required inputs:** Prepared Baček HDF5; 04 foundational results ZIP.

**Outputs:** Anatomical-group predictions and metrics.

**Usage:** Run after 04. Files are read from the notebook’s current working directory unless specified in its configuration.


In [ ]:
# 1 — Imports, fixed source configuration and general helpers
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import io
import json
import os
import platform
import sys
import time
import zipfile

import h5py
import numpy as np
import scipy
import sklearn
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

H5 = Path('bacek_clean_common_angles_metabolic_dataset.h5')
REFERENCE_ZIP = Path('04_Bacek_Foundational_Predictive_Models_results.zip')
OUT = Path('07_Bacek_Anatomical_Gait_Information_results')
ARCHIVE = Path('07_Bacek_Anatomical_Gait_Information_results.zip')
SOURCE_SHA256 = 'c73ce561e94de52f1dc86128829c43ef536c4cf6b5f4aa9af1ea060409221fa3'
NOTEBOOK04_ZIP_SHA256 = 'd0d5fb49f3f767dc1b99c790afda9aad630bfae5199fced5c5512c14329c7752'
CHANNELS = ('hip_left', 'hip_right', 'knee_left', 'knee_right',
            'ankle_left', 'ankle_right', 'pelvis_sagittal',
            'pelvis_frontal', 'pelvis_transverse')
ALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0)
OUTER_FOLDS = {
    1: ('S08', 'S10', 'S16', 'S21'),
    2: ('S11', 'S13', 'S17', 'S20'),
    3: ('S01', 'S02', 'S05', 'S18'),
    4: ('S03', 'S07', 'S12', 'S14', 'S19'),
    5: ('S04', 'S06', 'S09', 'S15'),
}
EXPECTED_COUNTS = (55, 53, 56, 65, 55)
FOLD_OF = {person: fold for fold, people in OUTER_FOLDS.items() for person in people}
GROUP_CHANNELS = {
    'HIP_LR': (0, 1),
    'KNEE_LR': (2, 3),
    'ANKLE_LR': (4, 5),
    'PELVIS_3': (6, 7, 8),
    'ALL9': tuple(range(9)),
}
NEW_GROUPS = ('HIP_LR', 'KNEE_LR', 'ANKLE_LR', 'PELVIS_3')
ALL_GROUPS = NEW_GROUPS + ('ALL9',)


def check(condition, message):
    if not bool(condition):
        raise AssertionError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def from_zip_csv(archive, name):
    with archive.open(name) as handle:
        return list(csv.DictReader(io.TextIOWrapper(handle, encoding='utf-8')))


def save_csv(path, records, columns):
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(records)


def rmse(target, prediction):
    error = np.asarray(prediction, dtype=np.float64) - np.asarray(target, dtype=np.float64)
    return float(np.sqrt(np.mean(error**2, dtype=np.float64)))


def metrics(target, prediction):
    target = np.asarray(target, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    error = prediction - target
    sst = float(np.sum((target - target.mean())**2, dtype=np.float64))
    return {
        'n_trials': len(target), 'RMSE_W_per_kg': rmse(target, prediction),
        'MAE_W_per_kg': float(np.mean(np.abs(error), dtype=np.float64)),
        'bias_W_per_kg': float(np.mean(error, dtype=np.float64)),
        'R2': float(1.0 - np.sum(error**2, dtype=np.float64)/sst) if sst > 0 else None,
    }

print('Python:', platform.python_version(), '| NumPy:', np.__version__,
      '| scikit-learn:', sklearn.__version__)

In [ ]:
# 2 — Verify Notebook 04's reusable all-nine reference and saved partitions
check(REFERENCE_ZIP.is_file(), f'Missing Notebook 04 archive: {REFERENCE_ZIP.name}')
reference_zip_sha = sha256_file(REFERENCE_ZIP)
check(reference_zip_sha == NOTEBOOK04_ZIP_SHA256,
      'Notebook 04 archive fingerprint changed; verify the reference before continuing')

with zipfile.ZipFile(REFERENCE_ZIP) as archive:
    reference_summary = json.loads(archive.read('run_summary.json'))
    ref_ledger = from_zip_csv(archive, 'eligible_trial_ledger.csv')
    ref_outer = from_zip_csv(archive, 'outer_fold_manifest.csv')
    ref_inner = from_zip_csv(archive, 'inner_fold_manifest.csv')
    ref_inner_scores = from_zip_csv(archive, 'inner_candidate_scores.csv')
    ref_pooled_scores = from_zip_csv(archive, 'pooled_candidate_scores.csv')
    ref_selected = from_zip_csv(archive, 'selected_models.csv')
    ref_predictions = from_zip_csv(archive, 'held_out_predictions.csv')
    ref_timings = from_zip_csv(archive, 'outer_model_timings.csv')
    ref_checks = from_zip_csv(archive, 'verification_checks.csv')
    ref_metrics = from_zip_csv(archive, 'pooled_metrics.csv')

check(reference_summary['status'] == 'PASS' and
      reference_summary['source_sha256'] == SOURCE_SHA256 and
      reference_summary['total_primary_trials'] == 284,
      'Notebook 04 reference status/source/population mismatch')
check(ref_checks and all(r['status'] == 'PASS' for r in ref_checks),
      'Notebook 04 contains failed or missing verification checks')
check(len(ref_ledger) == 284 and len(ref_outer) == 5 and len(ref_inner) == 20,
      'Incomplete Notebook 04 trial or fold manifests')
ref_by_uid = {r['trial_uid']: r for r in ref_ledger}
check(len(ref_by_uid) == 284 and len({r['subject_id'] for r in ref_ledger}) == 21,
      'Duplicate or incomplete Notebook 04 trial identities')
check(sum(int(r['retained_cycles']) for r in ref_ledger) == 60757,
      'Notebook 04 primary cycle total mismatch')
check(all(float(r['observed_W_per_kg']) > 0 and
          np.isfinite(float(r['observed_W_per_kg'])) for r in ref_ledger),
      'Notebook 04 target validity mismatch')

ref_w_predictions = {r['trial_uid']: r for r in ref_predictions
                     if r['procedure'] == 'W-ALL9_FIXED'}
ref_w_selections = {int(r['outer_fold']): r for r in ref_selected
                    if r['procedure'] == 'W-ALL9_FIXED'}
ref_w_inner = [r for r in ref_inner_scores if r['representation'] == 'W-ALL9']
ref_w_pooled = [r for r in ref_pooled_scores if r['representation'] == 'W-ALL9']
check(len(ref_w_predictions) == 284 and set(ref_w_predictions) == set(ref_by_uid),
      'Notebook 04 W-ALL9 prediction coverage mismatch')
check(len(ref_w_selections) == 5 and len(ref_w_inner) == 120 and
      len(ref_w_pooled) == 30, 'Notebook 04 W-ALL9 selection evidence incomplete')

for ref in ref_w_predictions.values():
    trial = ref_by_uid[ref['trial_uid']]
    check(ref['subject_id'] == trial['subject_id'] and
          ref['trial_id'] == trial['trial_id'] and
          int(ref['outer_fold']) == int(trial['outer_fold']) and
          float(ref['observed_W_per_kg']) == float(trial['observed_W_per_kg']) and
          np.isfinite(float(ref['predicted_W_per_kg'])),
          f'Notebook 04 W-ALL9 prediction mismatch: {ref["trial_uid"]}')

for fold in OUTER_FOLDS:
    matches = [r for r in ref_outer if int(r['outer_fold']) == fold]
    check(len(matches) == 1, f'Missing/duplicate outer fold {fold}')
    r = matches[0]
    check(set(r['test_participants'].split(';')) == set(OUTER_FOLDS[fold]) and
          set(r['development_participants'].split(';')) == set(FOLD_OF) - set(OUTER_FOLDS[fold]) and
          int(r['n_test_trials']) == EXPECTED_COUNTS[fold - 1] and
          int(r['n_development_trials']) == 284-EXPECTED_COUNTS[fold - 1],
          f'Notebook 04 outer fold {fold} inconsistent')
    choices = [r for r in ref_w_pooled if int(r['outer_fold']) == fold]
    check(len(choices) == 6 and set(float(r['alpha']) for r in choices) == set(ALPHAS),
          f'Missing W-ALL9 candidate in fold {fold}')
    optimal = min(choices, key=lambda r: (float(r['inner_pooled_RMSE_W_per_kg']), float(r['alpha'])))
    recorded = ref_w_selections[fold]
    check(float(recorded['alpha']) == float(optimal['alpha']) and
          float(recorded['inner_pooled_RMSE_W_per_kg']) ==
          float(optimal['inner_pooled_RMSE_W_per_kg']) and
          int(recorded['n_features']) == 909,
          f'Notebook 04 all-nine selection does not match its inner scores: fold {fold}')

print('Verified Notebook 04 W-ALL9 predictions, six penalties per fold and saved partitions')

In [ ]:
# 3 — Read corrected HDF5 and construct trial-mean waveform features (no label leakage)
check(H5.is_file(), f'Missing corrected Baček HDF5: {H5.name}')
source_sha = sha256_file(H5)
check(source_sha == SOURCE_SHA256, 'Incorrect HDF5 fingerprint; stop execution')


def trial_mean_waveforms(trial):
    waveforms = []
    n_cycles = None
    for channel in CHANNELS:
        check(channel in trial, f'Missing angle channel: {channel}')
        angle = np.asarray(trial[channel][()], dtype=np.float64)
        check(angle.ndim == 2 and angle.shape[1] == 101 and
              np.isfinite(angle).all(), f'Invalid angle values/shape: {channel}')
        if n_cycles is None:
            n_cycles = angle.shape[0]
        check(angle.shape[0] == n_cycles, 'Inconsistent retained-cycle counts')
        waveforms.append(angle.mean(axis=0, dtype=np.float64))
    return np.stack(waveforms), n_cycles

records, waveforms = [], []
with h5py.File(H5, 'r') as h5:
    check(set(h5['subjects']) == set(FOLD_OF), 'Source participant identities mismatch')
    for person in sorted(h5['subjects']):
        for trial_id in sorted(h5['subjects'][person]['trials']):
            if not (trial_id.startswith('T') and trial_id[1:].isdigit() and
                    1 <= int(trial_id[1:]) <= 15):
                continue
            trial = h5['subjects'][person]['trials'][trial_id]
            uid = f'{person}_{trial_id}'
            m, count = trial_mean_waveforms(trial)
            for flag in ('valid', 'comparison_eligible', 'unrestricted_comparison_eligible'):
                check(flag in trial.attrs and
                      np.asarray(trial.attrs[flag]).reshape(-1)[0] == 1,
                      f'{uid}: invalid source flag {flag}')
            target = np.asarray(trial['metcost_mean'][()])
            check(target.size == 1 and np.issubdtype(target.dtype, np.number)
                  and np.isrealobj(target), f'{uid}: non-scalar/complex target')
            target_value = float(target.reshape(-1)[0])
            check(np.isfinite(target_value) and target_value > 0,
                  f'{uid}: nonfinite/nonpositive target')
            original_indices = np.asarray(trial['original_cycle_number'][()]).reshape(-1)
            check(len(original_indices) == count and np.isfinite(original_indices).all()
                  and np.all(original_indices == np.floor(original_indices))
                  and np.all(np.diff(original_indices) > 0),
                  f'{uid}: invalid cycle provenance')
            if uid == 'S06_T11':
                check(count == 85, 'S06_T11 must have 85 cycles')
                continue
            check(count >= 100, f'{uid}: unexpected eligibility')
            check(uid in ref_by_uid, f'{uid}: missing Notebook 04 trial')
            previous = ref_by_uid[uid]
            check(previous['subject_id'] == person and previous['trial_id'] == trial_id
                  and int(previous['outer_fold']) == FOLD_OF[person] and
                  int(previous['retained_cycles']) == count and
                  float(previous['observed_W_per_kg']) == target_value,
                  f'{uid}: source / Notebook 04 ledger mismatch')
            records.append({'trial_uid': uid, 'subject_id': person, 'trial_id': trial_id,
                            'outer_fold': FOLD_OF[person], 'retained_cycles': count,
                            'observed_W_per_kg': target_value})
            waveforms.append(m)

check(len(records) == 284 and set(r['trial_uid'] for r in records) == set(ref_by_uid),
      'Primary trial coverage differs from Notebook 04')
check(sum(r['retained_cycles'] for r in records) == 60757,
      'Primary cycle total differs from source protocol')
check([sum(r['outer_fold'] == fold for r in records) for fold in OUTER_FOLDS]
      == list(EXPECTED_COUNTS), 'Outer-fold trial totals mismatch')

M = np.stack(waveforms).astype(np.float64, copy=False)  # (284, 9, 101)
X = {name: M[:, indices, :].reshape(len(records), len(indices)*101)
     for name, indices in GROUP_CHANNELS.items()}
check(M.shape == (284, 9, 101) and X['ALL9'].shape == (284, 909),
      'Incorrect waveform array dimensions')
check(all(X[g].shape == (284, len(GROUP_CHANNELS[g])*101)
          and np.isfinite(X[g]).all() for g in ALL_GROUPS),
      'Anatomical group feature dimensions/values invalid')

y = np.asarray([r['observed_W_per_kg'] for r in records], dtype=np.float64)
groups = np.asarray([r['subject_id'] for r in records])
uids = np.asarray([r['trial_uid'] for r in records])
uid_index = {uid: i for i, uid in enumerate(uids)}
check(len(uid_index) == 284 and set(groups) == set(FOLD_OF), 'Incorrect trial/subject indexing')
print('Verified 284 trials / 21 participants; anatomical matrices:',
      {g: X[g].shape for g in ALL_GROUPS})

In [ ]:
# 4 — Reconstruct the exact saved inner training/validation partitions
inner_by_outer = {}
for fold in OUTER_FOLDS:
    development_indices = np.flatnonzero(~np.isin(groups, OUTER_FOLDS[fold]))
    test_indices = np.flatnonzero(np.isin(groups, OUTER_FOLDS[fold]))
    test_people = set(OUTER_FOLDS[fold])
    check(len(test_indices) == EXPECTED_COUNTS[fold-1] and
          set(groups[development_indices]).isdisjoint(test_people),
          f'Outer fold {fold} participant boundary incorrect')
    saved = [r for r in ref_inner if int(r['outer_fold']) == fold]
    check(len(saved) == 4, f'Missing saved inner folds for outer fold {fold}')
    saved = sorted(saved, key=lambda r: int(r['inner_fold']))
    inner_partitions, covered = [], []
    for i, r in enumerate(saved, 1):
        check(int(r['inner_fold']) == i, 'Unexpected inner fold numbering')
        train_people = set(r['train_participants'].split(';'))
        val_people = set(r['validation_participants'].split(';'))
        train_indices = np.flatnonzero(np.isin(groups, list(train_people)))
        val_indices = np.flatnonzero(np.isin(groups, list(val_people)))
        check(train_people and val_people and train_people.isdisjoint(val_people)
              and train_people | val_people == set(groups[development_indices])
              and train_people.isdisjoint(test_people)
              and val_people.isdisjoint(test_people),
              f'Outer {fold} inner {i}: participant overlap or omission')
        check(len(train_indices) == int(r['n_train_trials']) and
              len(val_indices) == int(r['n_validation_trials']),
              f'Outer {fold} inner {i}: saved count mismatch')
        inner_partitions.append((train_indices, val_indices))
        covered.extend(val_indices.tolist())
    check(sorted(covered) == sorted(development_indices.tolist()),
          f'Outer fold {fold}: inner trials not covered exactly once')
    inner_by_outer[fold] = (development_indices, test_indices, inner_partitions)

print('All 20 Notebook 04 inner folds verified and reused (no regenerated splits)')

## Anatomical-group Ridge evaluation


In [ ]:
# 5 — Fit and select four anatomical groups; reuse all-nine reference exactly

def ridge_model(alpha):
    return Pipeline([('scale', StandardScaler()),
                     ('ridge', Ridge(alpha=alpha, fit_intercept=True))])


def candidate_key(r):
    # Within one fixed group, exact pooled-RMSE ties favour smaller alpha.
    return (r['inner_pooled_RMSE_W_per_kg'], r['alpha'])

inner_scores, pooled_scores, selections = [], [], []
new_predictions, model_timings = [], []
started_at = datetime.now(timezone.utc).isoformat()
total_start = time.perf_counter()

for fold in OUTER_FOLDS:
    dev_idx, test_idx, inner_parts = inner_by_outer[fold]
    for group_name in NEW_GROUPS:
        matrix = X[group_name]
        group_scores = []
        for alpha in ALPHAS:
            pooled_inner = np.full(len(dev_idx), np.nan, dtype=np.float64)
            dev_lookup = {int(index): k for k, index in enumerate(dev_idx)}
            for inner_fold, (train_idx, val_idx) in enumerate(inner_parts, 1):
                model = ridge_model(alpha)
                tic = time.perf_counter()
                model.fit(matrix[train_idx], y[train_idx])
                fit_seconds = time.perf_counter() - tic
                fitted_train = np.asarray(model.predict(matrix[train_idx]), dtype=np.float64)
                fitted_val = np.asarray(model.predict(matrix[val_idx]), dtype=np.float64)
                check(np.isfinite(fitted_train).all() and np.isfinite(fitted_val).all(),
                      f'{group_name} fold {fold}: nonfinite inner predictions')
                for index, pred in zip(val_idx, fitted_val):
                    pooled_inner[dev_lookup[int(index)]] = pred
                inner_scores.append({
                    'outer_fold': fold, 'inner_fold': inner_fold, 'group_id': group_name,
                    'alpha': alpha, 'n_features': matrix.shape[1],
                    'n_train_trials': len(train_idx), 'n_validation_trials': len(val_idx),
                    'train_RMSE_W_per_kg': rmse(y[train_idx], fitted_train),
                    'validation_RMSE_W_per_kg': rmse(y[val_idx], fitted_val),
                    'fit_seconds': fit_seconds, 'source_run': 'NOTEBOOK07',
                })
            check(np.isfinite(pooled_inner).all(), 'Incomplete pooled inner predictions')
            score = {'outer_fold': fold, 'group_id': group_name, 'alpha': alpha,
                     'n_features': matrix.shape[1],
                     'inner_pooled_RMSE_W_per_kg': rmse(y[dev_idx], pooled_inner),
                     'source_run': 'NOTEBOOK07'}
            pooled_scores.append(score)
            group_scores.append(score)
        best = min(group_scores, key=candidate_key)
        selections.append({'outer_fold': fold, 'group_id': group_name,
                           'alpha': best['alpha'], 'n_features': best['n_features'],
                           'inner_pooled_RMSE_W_per_kg': best['inner_pooled_RMSE_W_per_kg'],
                           'source_run': 'NOTEBOOK07'})

        final_model = ridge_model(best['alpha'])
        tic = time.perf_counter()
        final_model.fit(matrix[dev_idx], y[dev_idx])
        fit_seconds = time.perf_counter() - tic
        tic = time.perf_counter()
        predicted = np.asarray(final_model.predict(matrix[test_idx]), dtype=np.float64)
        prediction_seconds = time.perf_counter() - tic
        check(len(predicted) == len(test_idx) and np.isfinite(predicted).all(),
              'Nonfinite/incomplete outer prediction')
        model_timings.append({
            'outer_fold': fold, 'group_id': group_name, 'alpha': best['alpha'],
            'n_features': best['n_features'], 'fit_seconds': fit_seconds,
            'prediction_seconds': prediction_seconds,
            'prediction_seconds_per_trial': prediction_seconds/len(test_idx),
            'n_train_trials': len(dev_idx), 'n_predicted_trials': len(test_idx),
            'source_run': 'NOTEBOOK07',
        })
        for index, prediction in zip(test_idx, predicted):
            record = records[index]
            new_predictions.append({
                'group_id': group_name, 'outer_fold': fold,
                'subject_id': record['subject_id'], 'trial_id': record['trial_id'],
                'trial_uid': record['trial_uid'],
                'observed_W_per_kg': record['observed_W_per_kg'],
                'predicted_W_per_kg': float(prediction), 'source_run': 'NOTEBOOK07',
            })
        print(f'Fold {fold}: {group_name:<9s} alpha={best["alpha"]:g}; '
              f'inner RMSE={best["inner_pooled_RMSE_W_per_kg"]:.5f}')

# Import every all-nine inner candidate score and the fixed-W selections.
for r in ref_w_inner:
    inner_scores.append({
        'outer_fold': int(r['outer_fold']), 'inner_fold': int(r['inner_fold']),
        'group_id': 'ALL9', 'alpha': float(r['alpha']), 'n_features': 909,
        'n_train_trials': int(r['n_train_trials']),
        'n_validation_trials': int(r['n_validation_trials']),
        'train_RMSE_W_per_kg': float(r['train_RMSE_W_per_kg']),
        'validation_RMSE_W_per_kg': float(r['validation_RMSE_W_per_kg']),
        'fit_seconds': float(r['fit_seconds']), 'source_run': 'NOTEBOOK04',
    })
for r in ref_w_pooled:
    pooled_scores.append({
        'outer_fold': int(r['outer_fold']), 'group_id': 'ALL9',
        'alpha': float(r['alpha']), 'n_features': 909,
        'inner_pooled_RMSE_W_per_kg': float(r['inner_pooled_RMSE_W_per_kg']),
        'source_run': 'NOTEBOOK04',
    })
for fold, r in sorted(ref_w_selections.items()):
    selections.append({
        'outer_fold': fold, 'group_id': 'ALL9', 'alpha': float(r['alpha']),
        'n_features': 909,
        'inner_pooled_RMSE_W_per_kg': float(r['inner_pooled_RMSE_W_per_kg']),
        'source_run': 'NOTEBOOK04',
    })
    reused_timing = [r for r in ref_timings
                     if int(r['outer_fold']) == fold
                     and r['representation'] == 'W-ALL9'
                     and float(r['alpha']) == float(ref_w_selections[fold]['alpha'])]
    check(len(reused_timing) == 1, f'Notebook 04 all-nine timing absent in fold {fold}')
    timing = reused_timing[0]
    model_timings.append({
        'outer_fold': fold, 'group_id': 'ALL9', 'alpha': float(timing['alpha']),
        'n_features': 909, 'fit_seconds': float(timing['fit_seconds']),
        'prediction_seconds': float(timing['prediction_seconds']),
        'prediction_seconds_per_trial': float(timing['prediction_seconds']) / int(timing['n_predicted_trials']),
        'n_train_trials': int(timing['n_train_trials']),
        'n_predicted_trials': int(timing['n_predicted_trials']),
        'source_run': 'NOTEBOOK04',
    })

elapsed = time.perf_counter() - total_start
print(f'New anatomical fits complete in {elapsed:.1f}s; all-nine reference reused')

In [ ]:
# 6 — Build complete matched prediction ledger and calculate descriptive comparisons
prediction_ledger = list(new_predictions)
for uid, ref in ref_w_predictions.items():
    prediction_ledger.append({
        'group_id': 'ALL9', 'outer_fold': int(ref['outer_fold']),
        'subject_id': ref['subject_id'], 'trial_id': ref['trial_id'],
        'trial_uid': uid, 'observed_W_per_kg': float(ref['observed_W_per_kg']),
        'predicted_W_per_kg': float(ref['predicted_W_per_kg']),
        'source_run': 'NOTEBOOK04',
    })

by_group_uid = {(r['group_id'], r['trial_uid']): r for r in prediction_ledger}
check(len(prediction_ledger) == 5*284 and len(by_group_uid) == 5*284,
      'Missing or duplicate anatomical held-out predictions')
check(all(set(r['trial_uid'] for r in prediction_ledger if r['group_id'] == g)
          == set(uids) for g in ALL_GROUPS), 'Anatomical groups use different test trial sets')
for r in prediction_ledger:
    source = records[uid_index[r['trial_uid']]]
    check(r['subject_id'] == source['subject_id'] and
          r['trial_id'] == source['trial_id'] and
          r['outer_fold'] == source['outer_fold'] and
          r['observed_W_per_kg'] == source['observed_W_per_kg'] and
          np.isfinite(r['predicted_W_per_kg']),
          f'Mismatched evaluation row: {r["group_id"]} {r["trial_uid"]}')

pooled_metrics, fold_metrics = [], []
for group_name in ALL_GROUPS:
    selected = [by_group_uid[(group_name, uid)] for uid in uids]
    pooled_metrics.append({'group_id': group_name, **metrics(
        [r['observed_W_per_kg'] for r in selected],
        [r['predicted_W_per_kg'] for r in selected])})
    for fold in OUTER_FOLDS:
        subset = [r for r in selected if r['outer_fold'] == fold]
        fold_metrics.append({'outer_fold': fold, 'group_id': group_name, **metrics(
            [r['observed_W_per_kg'] for r in subset],
            [r['predicted_W_per_kg'] for r in subset])})

metric_map = {r['group_id']: r for r in pooled_metrics}
paired_differences = []
for group_name in NEW_GROUPS:
    paired_differences.append({
        'reference_group': 'ALL9', 'candidate_group': group_name,
        'n_matched_trials': 284,
        'RMSE_reference_minus_candidate_W_per_kg':
            metric_map['ALL9']['RMSE_W_per_kg'] - metric_map[group_name]['RMSE_W_per_kg'],
        'MAE_reference_minus_candidate_W_per_kg':
            metric_map['ALL9']['MAE_W_per_kg'] - metric_map[group_name]['MAE_W_per_kg'],
    })

print('\nPooled held-out anatomical-group RMSE (W/kg):')
for r in pooled_metrics:
    print(f'  {r["group_id"]:<9s} {r["RMSE_W_per_kg"]:.6f}')
print('Paired contrasts are descriptive; participant-cluster intervals are deferred to E5.2.')

In [ ]:
# 7 — Final requirements-oriented verification checks
checks = []
def verify(name, outcome):
    checks.append({'check': name, 'status': 'PASS' if bool(outcome) else 'FAIL'})
    check(outcome, name)

verify('correct_source_and_reference_sha256',
       source_sha == SOURCE_SHA256 and reference_zip_sha == NOTEBOOK04_ZIP_SHA256)
verify('primary_trial_population_and_matching_reference',
       len(records) == 284 and set(uids) == set(ref_by_uid))
verify('anatomical_group_dimensions_and_finite_values',
       all(X[g].shape == (284, len(GROUP_CHANNELS[g])*101) and
           np.isfinite(X[g]).all() for g in ALL_GROUPS))
verify('fixed_5_outer_folds_correct',
       [sum(r['outer_fold'] == f for r in records) for f in OUTER_FOLDS]
       == list(EXPECTED_COUNTS))
verify('all_20_saved_inner_partitions_reused',
       len(inner_by_outer) == 5 and len(ref_inner) == 20 and
       all(len(inner_by_outer[f][2]) == 4 for f in OUTER_FOLDS))
verify('exactly_600_inner_score_records', len(inner_scores) == 5*5*6*4)
verify('exactly_150_pooled_candidate_scores', len(pooled_scores) == 5*5*6)
verify('exactly_25_selected_models', len(selections) == 5*5)
verify('exactly_20_new_outer_fits', len(new_predictions) == 4*284 and
       len([r for r in model_timings if r['source_run'] == 'NOTEBOOK07']) == 20)
verify('exactly_1420_unique_matched_predictions',
       len(prediction_ledger) == 5*284 and len(by_group_uid) == 5*284)
verify('all_prediction_values_finite',
       all(np.isfinite(r['predicted_W_per_kg']) for r in prediction_ledger))
verify('exactly_284_predictions_per_group',
       all(sum(r['group_id'] == g for r in prediction_ledger) == 284 for g in ALL_GROUPS))
verify('all_25_selections_minimize_pooled_inner_rmse', all(
       (selected['alpha'], selected['inner_pooled_RMSE_W_per_kg']) ==
       (best['alpha'], best['inner_pooled_RMSE_W_per_kg'])
       for selected in selections
       for best in [min([r for r in pooled_scores
                         if r['outer_fold'] == selected['outer_fold'] and
                         r['group_id'] == selected['group_id']], key=candidate_key)]))
verify('all9_predictions_reused_exactly', all(
       by_group_uid[('ALL9', uid)]['predicted_W_per_kg'] ==
       float(ref_w_predictions[uid]['predicted_W_per_kg']) for uid in uids))
verify('all9_selected_penalties_reused', all(
       selected['alpha'] == float(ref_w_selections[selected['outer_fold']]['alpha'])
       for selected in selections if selected['group_id'] == 'ALL9'))
verify('all9_metric_recalculation_matches_notebook04', all(
       abs(float(next(row[field] for row in ref_metrics
                      if row['procedure'] == 'W-ALL9_FIXED')) - metric_map['ALL9'][field]) < 1e-12
       for field in ('RMSE_W_per_kg', 'MAE_W_per_kg', 'bias_W_per_kg', 'R2')))
verify('all_comparison_metrics_finite', all(
       np.isfinite(r['RMSE_W_per_kg']) and np.isfinite(r['MAE_W_per_kg']) and
       np.isfinite(r['bias_W_per_kg']) and np.isfinite(r['R2']) for r in pooled_metrics))
verify('all_four_paired_differences_calculated',
       len(paired_differences) == 4 and all(r['n_matched_trials'] == 284 for r in paired_differences))

print('Verification:', len(checks), 'of', len(checks), 'checks passed')

In [ ]:
# 8 — Export machine-readable evidence and ZIP (no raw gait matrices)
OUT.mkdir(parents=True, exist_ok=True)

notebook_filename = '07_Bacek_Anatomical_Gait_Information.ipynb'
code_sha = None
if Path(notebook_filename).is_file():
    notebook_document = json.loads(Path(notebook_filename).read_text(encoding='utf-8'))
    combined_code = ''.join(''.join(c['source']) for c in notebook_document['cells']
                            if c['cell_type'] == 'code')
    code_sha = hashlib.sha256(combined_code.encode('utf-8')).hexdigest()

save_csv(OUT/'eligible_trial_ledger.csv', records,
         ['trial_uid','subject_id','trial_id','outer_fold','retained_cycles','observed_W_per_kg'])
save_csv(OUT/'inner_fold_manifest.csv', ref_inner,
         ['outer_fold','inner_fold','train_participants','validation_participants',
          'outer_test_participants','n_train_trials','n_validation_trials'])
save_csv(OUT/'anatomical_group_definitions.csv', [
    {'group_id': g, 'channel_names': ';'.join(CHANNELS[i] for i in GROUP_CHANNELS[g]),
     'ordered_channel_indices_0_based': ';'.join(str(i) for i in GROUP_CHANNELS[g]),
     'n_features': len(GROUP_CHANNELS[g])*101, 'waveform_encoding': 'trial mean, 101 phase samples',
     'estimator': 'StandardScaler -> Ridge'} for g in ALL_GROUPS],
     ['group_id','channel_names','ordered_channel_indices_0_based','n_features',
      'waveform_encoding','estimator'])
save_csv(OUT/'inner_candidate_scores.csv', inner_scores,
         ['outer_fold','inner_fold','group_id','alpha','n_features','n_train_trials',
          'n_validation_trials','train_RMSE_W_per_kg','validation_RMSE_W_per_kg',
          'fit_seconds','source_run'])
save_csv(OUT/'pooled_candidate_scores.csv', pooled_scores,
         ['outer_fold','group_id','alpha','n_features','inner_pooled_RMSE_W_per_kg','source_run'])
save_csv(OUT/'selected_models.csv', selections,
         ['outer_fold','group_id','alpha','n_features','inner_pooled_RMSE_W_per_kg','source_run'])
save_csv(OUT/'held_out_predictions.csv', prediction_ledger,
         ['group_id','outer_fold','subject_id','trial_id','trial_uid',
          'observed_W_per_kg','predicted_W_per_kg','source_run'])
save_csv(OUT/'pooled_metrics.csv', pooled_metrics,
         ['group_id','n_trials','RMSE_W_per_kg','MAE_W_per_kg','bias_W_per_kg','R2'])
save_csv(OUT/'outer_fold_metrics.csv', fold_metrics,
         ['outer_fold','group_id','n_trials','RMSE_W_per_kg','MAE_W_per_kg','bias_W_per_kg','R2'])
save_csv(OUT/'paired_rmse_differences.csv', paired_differences,
         ['reference_group','candidate_group','n_matched_trials',
          'RMSE_reference_minus_candidate_W_per_kg','MAE_reference_minus_candidate_W_per_kg'])
save_csv(OUT/'outer_model_timings.csv', model_timings,
         ['outer_fold','group_id','alpha','n_features','fit_seconds','prediction_seconds',
          'prediction_seconds_per_trial','n_train_trials','n_predicted_trials','source_run'])
save_csv(OUT/'verification_checks.csv', checks, ['check','status'])

summary = {
    'status': 'PASS' if all(r['status'] == 'PASS' for r in checks) else 'REVIEW_REQUIRED',
    'experiment': 'E3.1',
    'source_hdf5': H5.name, 'source_sha256': source_sha,
    'notebook04_results_filename': REFERENCE_ZIP.name,
    'notebook04_results_zip_sha256': reference_zip_sha,
    'notebook04_code_sha256': reference_summary['notebook_code_sha256'],
    'primary_population_trials': 284, 'participants': 21,
    'primary_cycles': 60757,
    'outcome': 'corrected stored metcost_mean; net W/kg',
    'evaluation': 'new person, fixed five outer participant folds; previously inspected, internally held-out',
    'inner_partitions': 'exact Notebook 04 saved four-fold participant partitions',
    'candidate_groups': {g: [CHANNELS[i] for i in GROUP_CHANNELS[g]] for g in ALL_GROUPS},
    'alpha_grid': list(ALPHAS),
    'pipeline': 'trial mean -> selected anatomical waveform columns -> StandardScaler -> Ridge(fit_intercept=True)',
    'selection': 'pooled inner trial-level RMSE; exact ties smaller alpha within group',
    'reuse': {'ALL9': 'W-ALL9_FIXED from Notebook 04: scores, selections, predictions and timing'},
    'n_new_held_out_predictions': len(new_predictions),
    'n_total_matched_held_out_predictions': len(prediction_ledger),
    'bootstrap_included': False,
    'started_at_utc': started_at,
    'finished_at_utc': datetime.now(timezone.utc).isoformat(),
    'new_model_execution_seconds': elapsed,
    'software': {
        'python': platform.python_version(), 'numpy': np.__version__,
        'scipy': scipy.__version__, 'scikit_learn': sklearn.__version__,
        'h5py': h5py.__version__, 'os': platform.platform(),
        'cpu_count': os.cpu_count(), 'kernel_python_executable': sys.executable,
    },
    'notebook_filename': notebook_filename,
    'notebook_code_sha256': code_sha,
    'code_fingerprint_basis': 'concatenated code-cell sources, UTF-8',
    'verification': {r['check']: r['status'] for r in checks},
}
(OUT/'run_summary.json').write_text(json.dumps(summary, indent=2, allow_nan=False), encoding='utf-8')

output_files = (
    'eligible_trial_ledger.csv','inner_fold_manifest.csv','anatomical_group_definitions.csv',
    'inner_candidate_scores.csv','pooled_candidate_scores.csv','selected_models.csv',
    'held_out_predictions.csv','pooled_metrics.csv','outer_fold_metrics.csv',
    'paired_rmse_differences.csv','outer_model_timings.csv','verification_checks.csv',
    'run_summary.json',
)
with zipfile.ZipFile(ARCHIVE, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as output_zip:
    for name in output_files:
        output_zip.write(OUT/name, arcname=name)
print('Saved', ARCHIVE, 'with', len(output_files), 'evidence files')
print('Recorded verification status:', summary['status'])

## Comparison interpretation
